# Differential expression analysis with pyDESeq2 - A549 LOY vs ROY
The script is based on the plot_minimal_pydeseq2_pipeline.py

### Import

In [1]:
# Import relevant packages
import os
import pickle as pkl

import anndata as ad
import numpy as np
import pandas as pd
import sklearn as sk
import scipy as sc 
import matplotlib.pyplot as plt
import seaborn as sns

from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference
from pydeseq2.ds import DeseqStats

In [23]:
# Set directories
data_dir = '/data'

In [ ]:
# import combined count and tpm matrix
df = pd.read_csv(os.path.join(data_dir,'Table3.csv'), sep=',', header=0, index_col=None) # Table 3 is provided as a supplementary table

### Single factor analysis
Differential expression analysis between A549_ROY vs A549_LOY group

In [ ]:
# identify count columns
count_cols = [c for c in df.columns if c.endswith('_count')]

# create a matrix for heatmap (genes as rows and samples as columns)
df_counts = df.set_index('gene_id')[count_cols]

# Convert the values in the DataFrame to integers
a549_df = df_counts.astype(int)

# Transpose dataframe
a549_df_t = a549_df.transpose()


In [ ]:
# create metadata dataframe
metadata_a549 = pd.DataFrame({
    'sample_id': a549_df_t.index,
})
# add group column (LOY or ROY)
metadata_a549['group'] = metadata_a549['sample_id'].apply(lambda x: 'LOY' if '_LOY' in x else 'ROY')

In [ ]:
# Ensure the indices of counts and metadata match
metadata_a549 = metadata_a549.reindex(a549_df_t.index)

print(a549_df_t.index)
print(metadata_a549.index)

In [ ]:
# Set the class 'DeseqDataSet', it has a count and a metadata argument, it is an AnnData object
# the design_factor specifies the columns used to compare samples
inference = DefaultInference(n_cpus=8)

dds_single_a549 = DeseqDataSet(
    counts=a549_df_t,
    metadata=metadata_a549,
    design_factors="group",
    refit_cooks=True,
    inference=inference,
)

In [ ]:
# After dds was initialized we can run the deseq2() method to fit dispersion and LFCs
dds_single_a549.deseq2()

### Statistical analysis after single-factor analysis

In [18]:
# statistics - This is the role of the :class:`DeseqStats` class. It has a unique mandatory argument,
# ``dds``, which should be a *fitted* :class:`DeseqDataSet <pydeseq2.dds.DeseqDataSet>`object.

# It also has a set of optional keyword arguments (see the :doc:`API documentation
# </api/docstrings/pydeseq2.ds.DeseqStats>`), among which:
#
# - ``alpha``: the p-value and adjusted p-value significance threshold (``0.05``
#   by default),
# - ``cooks_filter``: whether to filter p-values based on cooks outliers
#   (``True`` by default),
# - ``independent_filter``: whether to perform independent filtering to correct
#   p-value trends (``True`` by default).

stat_res_a549 = DeseqStats(dds_single_a549, inference=inference)

In [ ]:
# PyDESeq2 computes p-values using Wald tests. This can be done using the
# :meth:`summary() <DeseqStats.summary>` method, which runs the whole statistical
# analysis, cooks filtering and multiple testing adjustement included.  
# the results are stored in the ``results_df`` attribute (``stat_res.results_df``).

stat_res_a549.summary()

In [5]:
# Extract results (log2FoldChange and padj)
results_df_a549 = stat_res_a549.results_df
results_a549 = results_df_a549[['log2FoldChange', 'padj']]

# Add -log10(padj) for the volcano plot
results_a549['-log10(padj)'] = -np.log10(results_a549['padj'])

# Reverse the log2FC for plotting (because at the moment it is ROY vs LOY but we want LOY vs ROY)
results_a549['log2FoldChange'] = -results_a549['log2FoldChange']

### Volcano plot - Fig2C

In [16]:
# Either continue with results_a549 dataframe or with initial dataframe that already contains log2FC and p-values (for LOY vs ROY)
# filter initial df and exclude Y linked genes
exclude_Y = ['KDM5D', 'RPS4Y1', 'RPS4Y2', 'SRY', 'ZFY', 'TSPY1', 'DDX3Y', 'USP9Y', 'UTY', 'BPY2', 'DAZ1-4', 'CDY1', 'CDY2', 'PRKY', 
           'EIF1AY', 'TMSB4Y', 'VCY', 'NLGN4Y', 'AMELY', 'RBMY1A', 'TXLNGY', 'TBL1Y', 'PCDH11Y', 'FAM224B', 'XGY2']

df_noY = df[~df['gene_name'].isin(exclude_Y)].copy()

In [ ]:
# Volcano plot (Fig 2C)

df_noY["significant"] = (
    (df_noY["padj"] < 0.1 ) &
    (df_noY["log2FoldChange"].abs() > 0.5)
)

# Top 10 upregulated
top_up = df_noY[df_noY["significant"] & (df_noY["log2FoldChange"] > 0)].nlargest(7, "log2FoldChange")
# Top 10 downregulated
top_down = df_noY[df_noY["significant"] & (df_noY["log2FoldChange"] < 0)].nsmallest(7, "log2FoldChange")
# Top 20 based on padj
top_padj = df_noY[df_noY["significant"]].nsmallest(16, "padj")
# Combine for labeling
top_genes = pd.concat([top_up, top_down, top_padj]).drop_duplicates()

sns.set_theme(style="whitegrid")
plt.figure(figsize=(7, 6))

df_noY["color"] = "not deregulated"
df_noY.loc[(df["log2FoldChange"] >= 0.5) & (df_noY["padj"] < 0.1), "color"] = 'up in LOY'
df_noY.loc[(df["log2FoldChange"] <= -0.5) & (df_noY["padj"] < 0.1), "color"] = 'down in LOY'

sns.scatterplot(
    data=df_noY,
    x="log2FoldChange",
    y="#NAME?", #-log10(padj)=#NAME? column
    hue="color",
    palette={'not deregulated': '#B0B0B0', 'down in LOY': '#ED7D31', 'up in LOY': '#0000C0'},
    edgecolor=None,
    s=8,
    alpha=0.6
)
for _, row in top_genes.iterrows():
    plt.text(row["log2FoldChange"], row["#NAME?"], row["gene_name"], fontsize = 7) #-log10(padj)=#NAME? column

plt.axvline(0.5, linestyle="--", color="black", linewidth=0.8)
plt.axvline(-0.5, linestyle="--", color="black", linewidth=0.8)
plt.axhline(1, linestyle="--", color="black", linewidth=0.8) 

plt.xlabel("log2 Fold Change")
plt.ylabel(r"$-\log_{10}(\mathrm{adjusted\ p\text{-}value})$")
plt.xticks
plt.yticks
plt.legend(loc="upper right")

plt.tight_layout()
plt.show()